In [0]:
# Databricks notebook source

# COMMAND ----------
# 1. IMPORTS

from pyspark.sql import functions as F
from pyspark.ml.feature import Bucketizer

import math
import uuid


# COMMAND ----------
# 2. CONFIGURATION

# Historical data that champion v2 learned from
REFERENCE_TABLE = "fraud_mlops.gold.fraud_training_snapshot"

# Current 10K intentionally drifted production window
CURRENT_TABLE = "fraud_mlops.gold.production_drift_reference"

OUTPUT_TABLE = "fraud_mlops.gold.feature_drift_metrics"


NUMERIC_FEATURES = [
    "amount",
    "log_amount",
    "origin_txn_count_prior",
    "origin_avg_amount_prior",
    "origin_total_amount_prior",
    "amount_vs_origin_avg",
]


BINARY_FEATURES = [
    "type_cash_in",
    "type_cash_out",
    "type_debit",
    "type_payment",
    "type_transfer",
    "origin_has_history",
]


FEATURES = NUMERIC_FEATURES + BINARY_FEATURES


# PSI interpretation
PSI_LOW_THRESHOLD = 0.10
PSI_HIGH_THRESHOLD = 0.25

# Our project drift trigger
DRIFT_THRESHOLD = 0.20

EPSILON = 1e-6


# COMMAND ----------
# 3. LOAD REFERENCE AND CURRENT DATA

reference_df = (
    spark.table(REFERENCE_TABLE)
    .filter(F.col("dataset_split") == "train")
    .select(*FEATURES)
)


current_df = (
    spark.table(CURRENT_TABLE)
    .select(*FEATURES)
)


reference_total = reference_df.count()
current_total = current_df.count()


print("Training reference rows:", reference_total)
print("Current production rows:", current_total)


if reference_total == 0 or current_total == 0:
    raise ValueError(
        "Reference or current dataset is empty."
    )


# COMMAND ----------
# 4. COMMON PSI CALCULATION

def calculate_psi_from_counts(
    reference_counts,
    current_counts,
    join_column
):

    distribution = (
        reference_counts

        .join(
            current_counts,
            on=join_column,
            how="full"
        )

        .fillna(0)

        .withColumn(
            "reference_pct",
            F.greatest(
                F.col("reference_count") / reference_total,
                F.lit(EPSILON)
            )
        )

        .withColumn(
            "current_pct",
            F.greatest(
                F.col("current_count") / current_total,
                F.lit(EPSILON)
            )
        )

        .withColumn(
            "psi_component",
            (
                F.col("current_pct")
                - F.col("reference_pct")
            )
            *
            F.log(
                F.col("current_pct")
                / F.col("reference_pct")
            )
        )
    )


    psi_value = (
        distribution
        .agg(
            F.sum("psi_component")
            .alias("psi")
        )
        .first()["psi"]
    )


    return float(psi_value or 0.0)


# COMMAND ----------
# 5. NUMERIC PSI

def numeric_psi(feature):

    # Decile boundaries are learned ONLY from
    # historical champion-v2 training data
    quantiles = reference_df.approxQuantile(
        feature,
        [
            0.1,
            0.2,
            0.3,
            0.4,
            0.5,
            0.6,
            0.7,
            0.8,
            0.9
        ],
        0.01
    )


    quantiles = sorted(
        set(
            float(x)
            for x in quantiles
            if x is not None
            and math.isfinite(x)
        )
    )


    if not quantiles:
        return 0.0


    splits = [
        float("-inf"),
        *quantiles,
        float("inf")
    ]


    bucketizer = Bucketizer(
        splits=splits,
        inputCol=feature,
        outputCol="bucket"
    )


    reference_counts = (
        bucketizer
        .transform(
            reference_df.select(feature)
        )
        .groupBy("bucket")
        .count()
        .withColumnRenamed(
            "count",
            "reference_count"
        )
    )


    current_counts = (
        bucketizer
        .transform(
            current_df.select(feature)
        )
        .groupBy("bucket")
        .count()
        .withColumnRenamed(
            "count",
            "current_count"
        )
    )


    return calculate_psi_from_counts(
        reference_counts,
        current_counts,
        "bucket"
    )


# COMMAND ----------
# 6. BINARY PSI

def binary_psi(feature):

    reference_counts = (
        reference_df
        .groupBy(feature)
        .count()
        .withColumnRenamed(
            "count",
            "reference_count"
        )
    )


    current_counts = (
        current_df
        .groupBy(feature)
        .count()
        .withColumnRenamed(
            "count",
            "current_count"
        )
    )


    return calculate_psi_from_counts(
        reference_counts,
        current_counts,
        feature
    )


# COMMAND ----------
# 7. CALCULATE DRIFT FOR ALL 12 FEATURES

results = []


for feature in FEATURES:

    if feature in BINARY_FEATURES:
        psi = binary_psi(feature)
        feature_type = "binary"

    else:
        psi = numeric_psi(feature)
        feature_type = "numeric"


    results.append(
        (
            feature,
            feature_type,
            float(psi)
        )
    )


    print(
        f"{feature}: {psi:.6f}"
    )


# COMMAND ----------
# 8. CREATE RESULTS DATAFRAME

drift_df = spark.createDataFrame(
    results,
    [
        "feature",
        "feature_type",
        "psi"
    ]
)


# COMMAND ----------
# 9. ADD DRIFT STATUS

drift_df = (
    drift_df

    .withColumn(
        "drift_status",

        F.when(
            F.col("psi") < PSI_LOW_THRESHOLD,
            F.lit("LOW")
        )

        .when(
            F.col("psi") < PSI_HIGH_THRESHOLD,
            F.lit("MODERATE")
        )

        .otherwise(
            F.lit("HIGH")
        )
    )

    .withColumn(
        "is_drift",

        F.col("psi")
        >= F.lit(DRIFT_THRESHOLD)
    )
)


# COMMAND ----------
# 10. OVERALL DRIFT RESULT

drifted_features = (
    drift_df
    .filter(F.col("is_drift") == True)
    .count()
)


overall_drift = drifted_features > 0


print(
    "Drifted features:",
    drifted_features
)

print(
    "OVERALL DRIFT:",
    overall_drift
)


# COMMAND ----------
# 11. ADD MONITORING METADATA

monitoring_run_id = str(uuid.uuid4())


drift_df = (
    drift_df

    .withColumn(
        "monitoring_run_id",
        F.lit(monitoring_run_id)
    )

    .withColumn(
        "overall_drift",
        F.lit(overall_drift)
    )

    .withColumn(
        "reference_table",
        F.lit(REFERENCE_TABLE)
    )

    .withColumn(
        "current_table",
        F.lit(CURRENT_TABLE)
    )

    .withColumn(
        "reference_row_count",
        F.lit(reference_total)
    )

    .withColumn(
        "current_row_count",
        F.lit(current_total)
    )

    .withColumn(
        "calculated_at",
        F.current_timestamp()
    )
)


# COMMAND ----------
# 12. DISPLAY RESULTS

display(
    drift_df
    .orderBy(
        F.desc("psi")
    )
)


# COMMAND ----------
# 13. SAVE MONITORING HISTORY

(
    drift_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(OUTPUT_TABLE)
)


# COMMAND ----------
# 14. COMPLETION MESSAGE

print(
    f"Drift monitoring completed successfully.\n"
    f"Run ID: {monitoring_run_id}\n"
    f"Reference rows: {reference_total}\n"
    f"Current rows: {current_total}\n"
    f"Drifted features: {drifted_features}\n"
    f"Overall drift: {overall_drift}\n"
    f"Output table: {OUTPUT_TABLE}"
)